# Transformer-Based Chatbot (Decoder-Only Architecture)
We’ll break it into **components**:

* Tokenization
* Dataset preparation
* Decoder-only Transformer (from scratch)
* Training loop
* Text generation (inference)

---

# **1️⃣ Setup**

```bash
pip install torch tqdm
```

We’ll only use **PyTorch and tqdm** (for progress bars). No huggingface or pretrained models.

---


In [ ]:
# The following commands were used to try and resolve datasets import issues.
# They have already been executed.
# !pip uninstall -y datasets
# !pip install datasets
import datasets
print(datasets.__version__)

4.0.0


# **2️⃣ Tokenization and Vocabulary**

For simplicity, we’ll use a **character-level tokenizer**. You can extend to word-level later.

---

In [ ]:
import torch

class CharTokenizer:
    def __init__(self, text):
        chars = sorted(list(set(text)))
        self.stoi = {ch:i for i,ch in enumerate(chars)}
        self.itos = {i:ch for i,ch in enumerate(chars)}
        self.vocab_size = len(chars)

    def encode(self, text):
        return [self.stoi[ch] for ch in text]

    def decode(self, tokens):
        return "".join([self.itos[t] for t in tokens])

# **3️⃣ Dataset Preparation**

We’ll create sequences for autoregressive training.

In [ ]:
# from datasets import load_dataset

# # # Login using e.g. `huggingface-cli login` to access this dataset
# ds = load_dataset("pixelsandpointers/better_daily_dialog")

In [ ]:
# print(f'Dataset Type: {type(ds)}')
# print(f'Dataset Keys: {ds.keys()}')
# print(f'Dataset Features: {ds["train"].features}')

In [ ]:
# len(ds['train']['utterance'])

In [ ]:
# from torch.utils.data import Dataset, DataLoader

# class ChatDataset(Dataset):
#     def __init__(self, text, tokenizer, seq_len=64):
#         self.tokenizer = tokenizer
#         self.seq_len = seq_len
#         self.data = tokenizer.encode(text)

#     def __len__(self):
#         return len(self.data) - self.seq_len

#     def __getitem__(self, idx):
#         x = torch.tensor(self.data[idx:idx+self.seq_len])
#         y = torch.tensor(self.data[idx+1:idx+self.seq_len+1])
#         return x, y

# # Example usage
# text = "<USER> hello! <BOT> hi! how can i help you? <USER> tell me a joke <BOT> why did the chicken cross the road?"
# tokenizer = CharTokenizer(text)
# dataset = ChatDataset(text, tokenizer, seq_len=32)
# loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import torch
from torch.utils.data import Dataset, DataLoader

# Load dataset
ds = load_dataset("pixelsandpointers/better_daily_dialog")

# ---- GROUP DIALOGS ----
def group_dialogs(split):
    grouped = defaultdict(list)
    for row in ds[split]:
        role = "<USER>" if row["turn_type"] == 0 else "<BOT>"
        grouped[row["dialog_id"]].append(f"{role} {row['utterance']}")
    return grouped

train_dialogs = group_dialogs("test")

# ---- MERGE TURNS INTO STRINGS ----
processed = [" ".join(turns) for turns in train_dialogs.values()]

# ---- FULL TRAIN TEXT ----
train_text = "\n".join(processed)

# ---- TRAINING DATASET ----
class ChatDataset(Dataset):
    def __init__(self, text, tokenizer, seq_len=64):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.data = tokenizer.encode(text)

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx+self.seq_len])
        y = torch.tensor(self.data[idx+1:idx+self.seq_len+1])
        return x, y

tokenizer = CharTokenizer(train_text)
dataset = ChatDataset(train_text, tokenizer, seq_len=128)
loader = DataLoader(dataset, batch_size=256, shuffle=True, drop_last=True)

# **4️⃣ Decoder-Only Transformer Components**

We’ll implement **multi-head masked self-attention**, **feedforward**, **positional encoding**, and **decoder block**.


## **4.1 Positional Encoding**

In [ ]:
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

## **4.2 Masked Multi-Head Attention**

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.num_heads = num_heads

        self.qkv = nn.Linear(d_model, d_model*3)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()
        qkv = self.qkv(x)  # (B, L, 3*d_model)
        q, k, v = qkv.chunk(3, dim=-1)

        # Split heads
        q = q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1,2)  # (B, H, L, D)
        k = k.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1,2)
        v = v.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1,2)

        # Scaled dot-product attention with causal mask
        scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.d_head)  # (B,H,L,L)
        mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
        scores = scores.masked_fill(mask==0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)

        out = attn @ v  # (B,H,L,D)
        out = out.transpose(1,2).contiguous().view(batch_size, seq_len, d_model)
        out = self.fc_out(out)
        return out

## **4.3 Feedforward Layer**

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

## **4.4 Decoder Block**


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

## **4.5 Full Decoder-Only Transformer**


In [ ]:
class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2, d_ff=256, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.token_emb(x)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x)
        x = self.ln(x)
        logits = self.fc_out(x)
        return logits

# **5️⃣ Training Loop**


In [ ]:
import torch.optim as optim
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model = DecoderOnlyTransformer(vocab_size=tokenizer.vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, y in tqdm(loader):
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, tokenizer.vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

100%|██████████| 2113/2113 [02:13<00:00, 15.85it/s]


Epoch 1 Loss: 1.4417


100%|██████████| 2113/2113 [02:13<00:00, 15.84it/s]


Epoch 2 Loss: 1.1342


100%|██████████| 2113/2113 [02:12<00:00, 15.90it/s]


Epoch 3 Loss: 1.0801


100%|██████████| 2113/2113 [02:12<00:00, 15.95it/s]


Epoch 4 Loss: 1.0518


100%|██████████| 2113/2113 [02:12<00:00, 15.96it/s]

Epoch 5 Loss: 1.0333


# **6️⃣ Text Generation / Inference**


In [ ]:
def generate(model, tokenizer, prompt, max_len=100, temperature=1.0):
    model.eval()
    tokens = tokenizer.encode(prompt)
    tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_len):
        logits = model(tokens)
        next_token_logits = logits[0, -1, :] / temperature
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)
        if tokenizer.itos[next_token.item()] == "\n":  # optional EOS
            break

    return tokenizer.decode(tokens[0].tolist())

## **Test the Chatbot**


In [ ]:
prompt = "<USER> Hey man , you wanna buy some weed ? <BOT>"
response = generate(model, tokenizer, prompt, max_len=50)
print(response)

<USER> Hey man , you wanna buy some weed ? <BOT>  I help you , nice , only think of this will can 


# ✅ **Next Steps / Tips**

1. **Use a larger dataset** for meaningful dialogue.
2. **Increase d_model, layers, and heads** as GPU allows.
3. **Experiment with top-k / top-p sampling** for more creative responses.
4. **Add special tokens** like `<EOS>` and `<PAD>` for better sequence control.
5. **Keep context window** of last few exchanges for coherent conversation.

---

This is **100% custom** PyTorch code, and you **don’t need any pretrained model**. You now have a **decoder-only Transformer chatbot from scratch**.

---

# Save Model

In [ ]:
import json

torch.save(model.state_dict(), "decoder_transformer.pth")

with open("tokenizer.json", "w") as f:
    json.dump({
        "stoi": tokenizer.stoi,
        "itos": tokenizer.itos
    }, f)
